
# 01 — Retrieval from Qdrant for Financial QA

**Goal:** Hybrid retrieval (dense + sparse via RRF) from a Qdrant collection of 10-K annual report chunks, evaluated against the UDA benchmark.

**Pipeline stage:** Retrieval + LLM generation, evaluated against ground truth.

## Cell 1 — Setup & Connection Test

In [1]:
import os
import json
import pickle
import pandas as pd
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.models import (
    FieldCondition, Filter, MatchValue,
    NamedSparseVector, NamedVector,
    SearchParams, SparseVector,
    FusionQuery, Fusion, Prefetch, QueryRequest,
)
from FlagEmbedding import BGEM3FlagModel

# ── Config ──────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\phili\PycharmProjects\UDA")
QDRANT_PATH = str(PROJECT_ROOT / "data" / "qdrant_db")
COLLECTION = "annual_reports"
BGE_M3_MODEL = "BAAI/bge-m3"
TOP_K = 5  # retrieve wide, rerank later

# ── Connect to Qdrant (local/embedded mode) ─────────────
client = QdrantClient(path=QDRANT_PATH)

info = client.get_collection(COLLECTION)
print(f"Collection: {COLLECTION}")
print(f"Points:     {info.points_count}")
print(f"Vectors:    {list(info.config.params.vectors.keys())}")
print(f"Sparse:     {list(info.config.params.sparse_vectors.keys())}")
print(f"Status:     {info.status}")

C:\Users\phili\PycharmProjects\UDA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection: annual_reports
Points:     37600
Vectors:    ['dense']
Sparse:     ['sparse']
Status:     green


C:\Users\phili\AppData\Local\Temp\ipykernel_55788\3540312560.py:23: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <annual_reports> contains 37600 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path=QDRANT_PATH)


In [2]:
# ── Load BGE-M3 ────────────────────────────────────────
embed_model = BGEM3FlagModel(BGE_M3_MODEL, use_fp16=True)
print(f"BGE-M3 loaded (fp16, device: cuda)")

# Quick sanity check: encode a test query
test_out = embed_model.encode(
    ["What is the total revenue?"],
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
)
print(f"Dense dim:    {len(test_out['dense_vecs'][0])}")
print(f"Sparse nnz:   {len(test_out['lexical_weights'][0])}")
print("✓ Embedding model ready")

Fetching 30 files: 100%|██████████| 30/30 [00:00<?, ?it/s]


BGE-M3 loaded (fp16, device: cuda)


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Dense dim:    1024
Sparse nnz:   6
✓ Embedding model ready


## Cell 2 — Load QA Benchmark (fin only)

In [3]:

QA_DIR = PROJECT_ROOT / "data" / "qa"

df = pd.read_csv(QA_DIR / "fin_qa_filtered.csv", sep="|")
df["source"] = "fin"

# Extract ground-truth page from q_uid  (e.g. 'ADI/2009/page_49.pdf-1' → 'ADI/2009/page_49.pdf')
df["gt_pdf"] = df["q_uid"].str.rsplit("-", n=1).str[0]

print(f"Loaded {len(df)} questions (fin only)")
print(f"Documents: {sorted(df['doc_name'].unique())}")
print()
df[["doc_name", "q_uid", "gt_pdf", "question"]].head(10)

Loaded 365 questions (fin only)
Documents: ['AAL_2010', 'AAL_2013', 'AAL_2016', 'AAL_2017', 'AAPL_2002', 'AAPL_2003', 'AAPL_2004', 'AAPL_2005', 'AAPL_2006', 'AAPL_2007', 'AAPL_2008', 'AAPL_2010', 'AAPL_2011', 'AAPL_2012', 'AAPL_2014', 'AAPL_2015', 'AAPL_2016', 'AAPL_2017', 'AAP_2007', 'AAP_2011', 'AAP_2012', 'AAP_2013', 'AAP_2016', 'ABC_2005', 'ABMD_2003', 'ABMD_2004', 'ABMD_2005', 'ABMD_2006', 'ABMD_2007', 'ABMD_2009', 'ABMD_2011', 'ABMD_2012', 'ABMD_2015', 'ABMD_2018', 'ADBE_2003', 'ADBE_2008', 'ADBE_2009', 'ADBE_2011', 'ADBE_2012', 'CDNS_2016']



,doc_name,q_uid,gt_pdf,question
0,ABMD_2012,ABMD/2012/page_75.pdf-1,ABMD/2012/page_75.pdf,"during the 2012 year , did the equity awards i..."
1,ABMD_2012,ABMD/2012/page_75.pdf-2,ABMD/2012/page_75.pdf,for equity awards where the performance criter...
2,ABMD_2012,ABMD/2012/page_75.pdf-3,ABMD/2012/page_75.pdf,what is the net change in the number of shares...
3,ABMD_2012,ABMD/2012/page_75.pdf-4,ABMD/2012/page_75.pdf,what is the total value of vested shares durin...
4,ABMD_2012,ABMD/2012/page_41.pdf-2,ABMD/2012/page_41.pdf,did abiomed outperform the nasdaq medical equi...
5,ABMD_2012,ABMD/2012/page_41.pdf-1,ABMD/2012/page_41.pdf,did abiomed outperform the nasdaq composite in...
6,ABMD_2012,ABMD/2012/page_79.pdf-1,ABMD/2012/page_79.pdf,what was total rent expense for fiscal years 2...
7,ABMD_2012,ABMD/2012/page_79.pdf-2,ABMD/2012/page_79.pdf,how much of total future minimum lease payment...
8,ABMD_2012,ABMD/2012/page_79.pdf-3,ABMD/2012/page_79.pdf,what is the percentage increase in base rent f...
9,ABMD_2012,ABMD/2012/page_79.pdf-4,ABMD/2012/page_79.pdf,what is the percentage increase in base rent f...


## Cell 3 — `ask()`: Hybrid Retrieval + LLM Generation

**Filtering logic:** Every UDA benchmark question belongs to a specific document (`doc_name` = `TICKER_YEAR`).  
We **always** filter Qdrant by `pdf_name` so retrieval only searches within that document's chunks.  
This is mandatory for benchmark evaluation — without it, chunks from other companies/years could leak in.

In [4]:
import requests
import jsonlines
import re
from datetime import datetime

# ── LLM Config ─────────────────────────────────────────
LM_STUDIO_URL = "http://localhost:1234/v1/chat/completions"
LLM_MODEL = "qwen/qwq-32b"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def strip_think(text: str) -> str:
    """Remove <think>...</think> blocks from LLM output."""
    return re.sub(r"<think>[\s\S]*?</think>\s*", "", text).strip()


SYSTEM_PROMPT = """You are a financial analyst answering questions about corporate 10-K annual reports.
Answer ONLY based on the provided context.

## Rules
1. Chunks are ranked by retrieval score (Chunk 1 = best). On conflicting numbers, ALWAYS use the highest-ranked chunk. No deliberation.
2. Check section titles/page context for entity disambiguation. If the question names a specific entity, use that entity's data. If ambiguous, prefer the higher-ranked chunk.
3. Return ONLY the answer — no units, no currency symbols, no explanation.
   - Numbers: raw numeric value (e.g. 3.6, -24.5)
   - Ratios: decimals with 2 places. Percentages: 1 decimal (e.g. 19.9).
   - "By how much did X change" → percentage change.
   - Yes/no questions (e.g. "did X outperform Y"): answer 'yes' or 'no'.
4. "How much of X is Y" or "what portion" → return as percentage, not the absolute value.
5. "Total" means sum the components. "Average per year" means divide by the number of years.
6. Only respond 'INSUFFICIENT CONTEXT' if the metric is genuinely absent from ALL chunks.
7. If the answer requires combining values across chunks, compute and return the result."""


def ask(
    question: str,
    doc_name: str,
    top_k: int = TOP_K,
    temperature: float = 0.1,
    max_tokens: int = 12288,
) -> dict:
    """
    Hybrid retrieval (dense + sparse via RRF) from Qdrant,
    filtered to a specific document (pdf_name),
    then LLM generation via LM Studio.
    """

    # ── 1. Encode query ────────────────────────────────
    enc = embed_model.encode(
        [question],
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    dense_vec = enc["dense_vecs"][0].tolist()
    sparse_dict = enc["lexical_weights"][0]
    sparse_indices = [int(k) for k in sparse_dict.keys()]
    sparse_values = [float(v) for v in sparse_dict.values()]

    # ── 2. Mandatory document filter (pdf_name) ────────
    doc_filter = Filter(must=[
        FieldCondition(key="pdf_name", match=MatchValue(value=doc_name)),
    ])

    # ── 3. Hybrid retrieval: Prefetch + RRF fusion ─────
    results = client.query_points(
        collection_name=COLLECTION,
        prefetch=[
            Prefetch(
                query=dense_vec,
                using="dense",
                limit=top_k,
                filter=doc_filter,
            ),
            Prefetch(
                query=SparseVector(indices=sparse_indices, values=sparse_values),
                using="sparse",
                limit=top_k,
                filter=doc_filter,
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
    )

    # ── 4. Format chunks for context ───────────────────
    chunks = []
    context_parts = []

    for i, point in enumerate(results.points):
        p = point.payload

        # Use HTML for table chunks, text otherwise
        is_table = p.get("type") == "table"
        chunk_content = p.get("html", "") if is_table and p.get("html") else p.get("text", "")

        chunks.append({
            "chunk_id": str(point.id),
            "score": point.score,
            "pdf_name": p.get("pdf_name"),
            "page": int(p.get("page", 0)),
            "type": p.get("type"),
            "section": p.get("section"),
            "ticker": p.get("ticker"),
            "year": p.get("year"),
            "text": chunk_content,
        })

        header = (
            f"[Chunk {i+1}] ticker={p.get('ticker')} year={p.get('year')} "
            f"page={p.get('page')} section={p.get('section')} type={p.get('type')}"
        )
        context_parts.append(f"{header}\n{chunk_content}")

    context_str = "\n\n---\n\n".join(context_parts)

    # ── 5. Build user prompt ───────────────────────────
    user_prompt = f"Context:\n{context_str}\n\nQuestion: {question}\nAnswer:"

    # ── 6. Call LM Studio ──────────────────────────────
    resp = requests.post(
        LM_STUDIO_URL,
        json={
            "model": LLM_MODEL,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": temperature,
            "max_tokens": max_tokens,
        },
        timeout=500,
    )
    resp.raise_for_status()
    resp_json = resp.json()
    llm_answer = resp_json["choices"][0]["message"]["content"].strip()
    usage = resp_json.get("usage", {})

    # ── 7. Return everything ───────────────────────────
    return {
        "question": question,
        "doc_name": doc_name,
        "llm_answer": llm_answer,
        "prompt": user_prompt,
        "model": LLM_MODEL,
        "top_k": top_k,
        "retrieved_chunks": chunks,
        "context_chars": len(context_str),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
    }


def save_result(result: dict, q_uid: str, gt_answer_1: str, gt_answer_2: str,
                gt_pdf: str, run_file: Path,
                display_answer: str = None, think_text: str = None,
                retrieval_hit: bool = None, gt_page: int = None,
                retrieved_pages: list = None) -> None:
    """Append a single result as one JSONL line."""
    record = {
        "q_uid": q_uid,
        "gt_pdf": gt_pdf,
        "gt_answer_1": str(gt_answer_1),
        "gt_answer_2": str(gt_answer_2),
        "display_answer": display_answer,
        "think_text": think_text,
        "retrieval_hit": retrieval_hit,
        "gt_page": gt_page,
        "retrieved_pages": retrieved_pages,
        "timestamp": datetime.now().isoformat(),
        **result,
    }
    with jsonlines.open(run_file, mode="a") as writer:
        writer.write(record)


print("✓ ask(), strip_think(), save_result() defined")
print(f"  LLM:     {LLM_MODEL} @ {LM_STUDIO_URL}")
print(f"  Results: {RESULTS_DIR}")


✓ ask(), strip_think(), save_result() defined
  LLM:     qwen/qwq-32b @ http://localhost:1234/v1/chat/completions
  Results: C:\Users\phili\PycharmProjects\UDA\data\results


## Cell 4 — Run ALL Benchmark Questions

Iterates over all questions in `df`. Each question's `doc_name` (e.g. `ADI_2009`) is passed as the  
`pdf_name` filter → only that document's chunks are retrieved.

In [ ]:
# ── Run ALL benchmark questions ────────────────────────
from datetime import datetime

run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
run_file = RESULTS_DIR / f"run_fin_{run_ts}.jsonl"
run_json_file = RESULTS_DIR / f"run_fin_{run_ts}.json"

all_results = []   # collect for the per-run JSON

for i, row in df.iterrows():
    result = ask(
        question=row["question"],
        doc_name=row["doc_name"],
    )

    # ── Strip <think> for display ─────────────────────
    display_answer = strip_think(result["llm_answer"])
    think_match = re.search(r"<think>([\s\S]*?)</think>", result["llm_answer"])
    think_text = think_match.group(1).strip() if think_match else None

    # ── Retrieval hit: is GT page among retrieved? ────
    gt_page_match = re.search(r"page_(\d+)", str(row["gt_pdf"]))
    gt_page_num = int(gt_page_match.group(1)) if gt_page_match else None
    retrieved_pages = [c["page"] for c in result["retrieved_chunks"]]
    retrieval_hit = gt_page_num in retrieved_pages if gt_page_num is not None else None

    # ── Chunk type mix ────────────────────────────────
    type_counts = {}
    for c in result["retrieved_chunks"]:
        t = c["type"] or "unknown"
        type_counts[t] = type_counts.get(t, 0) + 1
    type_str = "  ".join(f"{k}={v}" for k, v in sorted(type_counts.items()))

    # ── Print diagnostics ─────────────────────────────
    hit_icon = "✓" if retrieval_hit else ("✗" if retrieval_hit is False else "?")
    print(f"── Q{i+1}/{len(df)}  doc={row['doc_name']} ──")
    print(f"Frage:          {row['question']}")
    print(f"Gold-Antwort:   {row['answer_1']}")
    print(f"LLM-Antwort:    {display_answer}")
    print(f"Retrieval hit:  {hit_icon}  (GT page={gt_page_num}, retrieved={retrieved_pages})")
    print(f"Chunk types:    {type_str}")
    print(f"Context:        {result['context_chars']:,} chars  |  prompt_tok={result.get('prompt_tokens')}  compl_tok={result.get('completion_tokens')}")

    # ── Print full retrieved chunks ───────────────────
    for ci, c in enumerate(result["retrieved_chunks"]):
        sec = (c['section'] or 'N/A')[:80]
        print(f"\n  ┌─ Chunk {ci+1}  score={c['score']:.4f}  doc={c['pdf_name']}  p.{c['page']}  {c['type']}  section={sec}")
        print(f"  │ {c['text']}")
        print(f"  └{'─'*60}")

    if think_text:
        print(f"\n  Think: {think_text}")
    print()

    # ── Build record ──────────────────────────────────
    record = {
        "q_uid": row["q_uid"],
        "gt_pdf": row["gt_pdf"],
        "gt_answer_1": str(row["answer_1"]),
        "gt_answer_2": str(row["answer_2"]),
        "display_answer": display_answer,
        "think_text": think_text,
        "retrieval_hit": retrieval_hit,
        "gt_page": gt_page_num,
        "retrieved_pages": retrieved_pages,
        **result,
    }

    # ── Save JSONL (streaming, one line per question) ─
    save_result(
        result=result,
        q_uid=row["q_uid"],
        gt_answer_1=row["answer_1"],
        gt_answer_2=row["answer_2"],
        gt_pdf=row["gt_pdf"],
        run_file=run_file,
        display_answer=display_answer,
        think_text=think_text,
        retrieval_hit=retrieval_hit,
        gt_page=gt_page_num,
        retrieved_pages=retrieved_pages,
    )

    all_results.append(record)

# ── Save per-run JSON with metadata ───────────────────
run_summary = {
    "run_id": run_ts,
    "model": LLM_MODEL,
    "top_k": TOP_K,
    "collection": COLLECTION,
    "num_questions": len(df),
    "timestamp": datetime.now().isoformat(),
    "results": all_results,
}

with open(run_json_file, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2, ensure_ascii=False)

print(f"\n✓ Done — {len(df)} questions")
print(f"  JSONL: {run_file}")
print(f"  JSON:  {run_json_file}")


── Q1/365  doc=ABMD_2012 ──
Frage:          during the 2012 year , did the equity awards in which the prescribed performance milestones were achieved exceed the equity award compensation expense for equity granted during the year?
Gold-Antwort:   yes
LLM-Antwort:    no
Retrieval hit:  ✓  (GT page=75, retrieved=[75, 75, 74, 68, 75])
Chunk types:    text=5
Context:        5,975 chars  |  prompt_tok=1783  compl_tok=5514

  ┌─ Chunk 1  score=1.0000  doc=ABMD_2012  p.75  text  section=Performance-Based Awards
  │ During the three months ended June 30, 2011, performance-based awards of restricted stock units for the potential issuance of 284,000 shares of common stock were issued to certain executive officers and members of the senior management, all of which would vest upon achievement of prescribed service milestones by the award recipients and revenue performance milestones by the Company. As of March 31, 2012, the Company determined that it met the prescribed targets for 184,000 shares u